In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys


parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    
from core_solver import (
    get_initial_conditions, 
    compute_two_fluid_properties, 
    factors_for_unit_conversion
)

# Professional Plotting Style
plt.rcParams.update({
    "text.usetex": True,                
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "figure.dpi": 150,                   
    "grid.alpha": 0.4,                    
})

# Universal Constants
hc = 197.3269804 # MeV fm
M_p = 1.2209e22  # MeV

In [2]:
def generate_mr_sequence(eos_path, factors, solver_func, lower=30, upper=10, samples=0.5, is_smooth=True):
    """
    Reads an EoS, extracts initial conditions, and computes the M-R sequence.
    Returns a DataFrame with the Mass and Radius for each central density.
    """
    df = pd.read_csv(eos_path)
    
    grid_p = df['Pressure'].values
    grid_e = df['Energy Density'].values
    grid_n = None

    if 'Number Density' in df.columns:
        grid_n = df['Number Density'].values

    p_init, e_init = get_initial_conditions(
        grid_p, grid_e, lower_trim=lower, upper_trim=upper, percen_samples=samples
    )
    print(f"[{eos_path.split('/')[-1]}] Extracted {len(p_init)} initial conditions.")

    radii, masses, n_particles, central_pressure = [], [], [], []
    for e_c in e_init:
        r0 = min(1e-6, 1e-3 * (1 / np.sqrt(e_c)))
        
        
        res = solver_func(
            e_c, None, 
            grid_e, grid_p, 
            None, None, 
            grid_n, None,  
            factors, r0=r0, save_profiles=False, is_smooth=is_smooth
        )
        
        radii.append(res['R_total_km'])
        masses.append(res['M_total_sol'])
        central_pressure.append(res['p_c_qm'])
        
        if grid_n is not None:
            n_particles.append(res['N_total'])
            
  
    output_data = {'Radius_km': radii, 'Mass_sol': masses, 'Central_Pressure_MeV_fm3': central_pressure}
    if grid_n is not None:
        output_data['Particles_fm_3'] = n_particles
        
    return pd.DataFrame(output_data)


In [3]:

B_val_MeV = 148.0
factors_bdm = factors_for_unit_conversion(B_val_MeV)


models_to_run = {
    "EoS MIT Bag Model (massive quarks)": {"path": "../data/eos_library/quark_matter_real.csv", "samples": 20, "upper": 0, "lower": 10},
    "EoS MIT Bag Model": {"path": "../data/eos_library/quark_matter_massless.csv", "samples": 20, "upper": 0, "lower": 10},
    "Bosonic DM $n=4$  $m=200$":  {"path": "../data/eos_library/bosonicDM_n_4_mb_200_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=200$":  {"path": "../data/eos_library/bosonicDM_n_40_mb_200_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=300$":  {"path": "../data/eos_library/bosonicDM_n_4_mb_300_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=300$":  {"path": "../data/eos_library/bosonicDM_n_40_mb_300_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=100$":  {"path": "../data/eos_library/bosonicDM_n_4_mb_100_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=100$":  {"path": "../data/eos_library/bosonicDM_n_40_mb_100_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=500$":  {"path": "../data/eos_library/bosonicDM_n_4_mb_500_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=500$":  {"path": "../data/eos_library/bosonicDM_n_40_mb_500_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=700$":  {"path": "../data/eos_library/bosonicDM_n_4_mb_700_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=700$":  {"path": "../data/eos_library/bosonicDM_n_40_mb_700_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=1000$": {"path": "../data/eos_library/bosonicDM_n_4_mb_1000_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=1000$": {"path": "../data/eos_library/bosonicDM_n_40_mb_1000_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=4$  $m=2000$": {"path": "../data/eos_library/bosonicDM_n_4_mb_2000_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
    "Bosonic DM $n=40$ $m=2000$": {"path": "../data/eos_library/bosonicDM_n_40_mb_2000_B_148.csv", "samples": 0.2, "upper": 20, "lower": 30},
  }




output_dir = "../data/mr_library"
os.makedirs(output_dir, exist_ok=True)

models_data = {}

for label, config in models_to_run.items():
    print(f"Processing {label}...")
    
    # Load raw EoS for the left plot
    eos_df = pd.read_csv(config["path"])
    
    # Compute M-R sequence for the right plot
    mr_df = generate_mr_sequence(
        eos_path=config["path"],
        factors=factors_bdm,
        solver_func=compute_two_fluid_properties, 
        samples=config["samples"], 
        upper=config["upper"], 
        lower=config["lower"],
        is_smooth=config.get("is_smooth", True)
    )
    
    # Store both DataFrames together for plotting later
    models_data[label] = {
        'eos': eos_df,
        'mr': mr_df
    }
    
    # ==========================================
    # NEW: Save the M-R sequence to a CSV file
    # ==========================================
    # Extract original filename (e.g., "bosonicDM_n_4_mb_300_B_148.csv")
    original_filename = os.path.basename(config["path"])
    
    # Prepend 'mr_' so we know it's the Mass-Radius output
    if not original_filename.startswith("mr_"):
        save_filename = f"mr_{original_filename}"
    else:
        save_filename = original_filename
        
    save_path = os.path.join(output_dir, save_filename)
    
    # Save without the pandas index column
    mr_df.to_csv(save_path, index=False)
    print(f"  -> Saved M-R data to {save_path}")

print("\nAll EoS and M-R sequences computed and saved successfully!")


Processing EoS MIT Bag Model (massive quarks)...
[quark_matter_real.csv] Extracted 200 initial conditions.
  -> Saved M-R data to ../data/mr_library/mr_quark_matter_real.csv
Processing EoS MIT Bag Model...
[quark_matter_massless.csv] Extracted 199 initial conditions.
  -> Saved M-R data to ../data/mr_library/mr_quark_matter_massless.csv
Processing Bosonic DM $n=4$  $m=200$...
[bosonicDM_n_4_mb_200_B_148.csv] Extracted 200 initial conditions.
  -> Saved M-R data to ../data/mr_library/mr_bosonicDM_n_4_mb_200_B_148.csv
Processing Bosonic DM $n=40$ $m=200$...
[bosonicDM_n_40_mb_200_B_148.csv] Extracted 200 initial conditions.
  -> Saved M-R data to ../data/mr_library/mr_bosonicDM_n_40_mb_200_B_148.csv
Processing Bosonic DM $n=4$  $m=300$...
[bosonicDM_n_4_mb_300_B_148.csv] Extracted 200 initial conditions.
  -> Saved M-R data to ../data/mr_library/mr_bosonicDM_n_4_mb_300_B_148.csv
Processing Bosonic DM $n=40$ $m=300$...
[bosonicDM_n_40_mb_300_B_148.csv] Extracted 200 initial conditions.
  